In [2]:
:dep zip = "0.6"

In [4]:
// :dep zip = "0.6"
use std::io::Read;

{
    // ファイルを読み込んでメモリ(Vec<u8>)に置くことで、
    // 所有権の複雑な問題を回避します
    let buf = std::fs::read("paper.docx").expect("File not found");
    let reader = std::io::Cursor::new(buf);
    let mut archive = zip::ZipArchive::new(reader).expect("Not a zip");

    // 1. 作成者(Author)を探す
    if let Ok(mut core_file) = archive.by_name("docProps/core.xml") {
        let mut s = String::new();
        core_file.read_to_string(&mut s).ok();
        println!("--- core.xml (メタデータ) ---");
        // <dc:creator>を探してください
        println!("{}\n", s);
    }

    // 2. 編集者(Last Modified By)を探す
    if let Ok(mut app_file) = archive.by_name("docProps/app.xml") {
        let mut s = String::new();
        app_file.read_to_string(&mut s).ok();
        println!("--- app.xml ---");
        println!("{}\n", s);
    }
}

--- core.xml (メタデータ) ---
<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<cp:coreProperties xmlns:cp="http://schemas.openxmlformats.org/package/2006/metadata/core-properties" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:dcterms="http://purl.org/dc/terms/" xmlns:dcmitype="http://purl.org/dc/dcmitype/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"><dc:title></dc:title><dc:subject></dc:subject><dc:creator></dc:creator><cp:keywords></cp:keywords><dc:description></dc:description><cp:lastModifiedBy></cp:lastModifiedBy><cp:revision>1</cp:revision><dcterms:created xsi:type="dcterms:W3CDTF">2012-09-19T19:32:00Z</dcterms:created><dcterms:modified xsi:type="dcterms:W3CDTF">2012-09-19T19:32:00Z</dcterms:modified></cp:coreProperties>

--- app.xml ---
<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<Properties xmlns="http://schemas.openxmlformats.org/officeDocument/2006/extended-properties" xmlns:vt="http://schemas.openxmlformats.org/officeDocument/2006/docPropsVTypes"><

()

In [7]:
use std::io::Read;

{
    let buf = std::fs::read("paper.docx").expect("File not found");
    let reader = std::io::Cursor::new(buf);
    let mut archive = zip::ZipArchive::new(reader).expect("Not a zip");

    // 本文が格納されているメインのXML
    if let Ok(mut doc_file) = archive.by_name("word/document.xml") {
        let mut s = String::new();
        doc_file.read_to_string(&mut s).ok();
        
        // 1. "vanish"（隠し設定）というキーワードを検索
        if s.contains("vanish") {
            println!("[!] 隠し文字(vanish)の設定が見つかりました！");
        }

        // 2. "Author" という文字列の前後を表示してみる
        if let Some(idx) = s.find("Author") {
            let start = if idx > 50 { idx - 50 } else { 0 };
            let end = std::cmp::min(idx + 150, s.len());
            println!("--- Author周辺のXML ---");
            println!("{}", &s[start..end]);
        } 
        else // ★見つからない場合は全文から名前っぽいのを探すため、タグを除去して表示（elseを削除してね！）
        {
            println!("--- 本文のテキスト抽出 (タグ除去) ---");
            let plain_text: String = s.chars().filter(|c| *c != '<' && *c != '>').collect(); // 簡易的な除去
            println!("{}", plain_text);
        }
    }
}

--- Author周辺のXML ---
:rPr><w:rFonts w:hint="eastAsia"/></w:rPr><w:t>No Author Given</w:t></w:r></w:p><w:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"><w:pPr><w:spacing w:line="360" w:lineRule="auto"/><


()

--- Author周辺のXML ---
:rPr><w:rFonts w:hint="eastAsia"/></w:rPr><w:t>No Author Given</w:t></w:r></w:p><w:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"><w:pPr><w:spacing w:line="360" w:lineRule="auto"/><
--- 本文のテキスト抽出 (タグ除去) ---
?xml version="1.0" encoding="UTF-8" standalone="yes"?
w:document xmlns:wpc="http://schemas.microsoft.com/office/word/2010/wordprocessingCanvas" xmlns:mc="http://schemas.openxmlformats.org/markup-compatibility/2006" xmlns:o="urn:schemas-microsoft-com:office:office" xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships" xmlns:m="http://schemas.openxmlformats.org/officeDocument/2006/math" xmlns:v="urn:schemas-microsoft-com:vml" xmlns:wp14="http://schemas.microsoft.com/office/word/2010/wordprocessingDrawing" xmlns:wp="http://schemas.openxmlformats.org/drawingml/2006/wordprocessingDrawing" xmlns:w10="urn:schemas-microsoft-com:office:word" xmlns:w="http://schemas.openxmlformats.org/wordprocessingml/2006/main" xmlns:w14="http://schemas.microsoft.com/office/word/2010/wordml" xmlns:wpg="http://schemas.microsoft.com/office/word/2010/wordprocessingGroup" xmlns:wpi="http://schemas.microsoft.com/office/word/2010/wordprocessingInk" xmlns:wne="http://schemas.microsoft.com/office/word/2006/wordml" xmlns:wps="http://schemas.microsoft.com/office/word/2010/wordprocessingShape" mc:Ignorable="w14 wp14"w:bodyw:p w:rsidR="00433D93" w:rsidRPr="000E7386" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"/w:jc w:val="center"/w:rPrw:sz w:val="48"//w:rPr/w:pPrw:bookmarkStart w:id="0" w:name="_GoBack"/w:bookmarkEnd w:id="0"/w:r w:rsidRPr="000E7386"w:rPrw:rFonts w:hint="eastAsia"/w:sz w:val="48"//w:rPrw:tP is equal to NP/w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRPr="00E22D59" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"/w:jc w:val="center"//w:pPr/w:pw:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"/w:jc w:val="center"//w:pPrw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:tNo Author Given/w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"/w:jc w:val="center"//w:pPrw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:tNo Institute Given/w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"//w:pPr/w:pw:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"//w:pPrw:r w:rsidRPr="00CE5FAA"w:rPrw:rFonts w:hint="eastAsia"/w:b//w:rPrw:tAbstract./w:t/w:rw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve" In this paper, we prove that the complexity classes P and NP are equal by showing that the 3-SAT problem can be solved in deterministic polynomial time./w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRDefault="002653CD" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"//w:pPrw:rw:rPrw:rFonts w:hint="eastAsia"/w:b/w:noProof/w:sz w:val="24"/w:szCs w:val="28"//w:rPrw:drawingwp:anchor distT="0" distB="0" distL="114300" distR="114300" simplePos="0" relativeHeight="251658240" behindDoc="0" locked="0" layoutInCell="1" allowOverlap="1" wp14:anchorId="2060EB98" wp14:editId="413AD905"wp:simplePos x="0" y="0"/wp:positionH relativeFrom="column"wp:posOffset3958590/wp:posOffset/wp:positionHwp:positionV relativeFrom="paragraph"wp:posOffset194945/wp:posOffset/wp:positionVwp:extent cx="1548765" cy="1409700"/wp:effectExtent l="0" t="0" r="0" b="0"/wp:wrapSquare wrapText="bothSides"/wp:docPr id="2" name="図 2" descr="C:\Users\FLxAG_XXXXXXXXXXXXXX\Documents\paper\P is equal to NP\remove_x\200px-Complexity_subsets_pspace.svg.png"/wp:cNvGraphicFramePra:graphicFrameLocks xmlns:a="http://schemas.openxmlformats.org/drawingml/2006/main" noChangeAspect="1"//wp:cNvGraphicFramePra:graphic xmlns:a="http://schemas.openxmlformats.org/drawingml/2006/main"a:graphicData uri="http://schemas.openxmlformats.org/drawingml/2006/picture"pic:pic xmlns:pic="http://schemas.openxmlformats.org/drawingml/2006/picture"pic:nvPicPrpic:cNvPr id="0" name="Picture 2" descr="C:\Users\FLxAG_XXXXXXXXXXXXXX\Documents\paper\P is equal to NP\remove_x\200px-Complexity_subsets_pspace.svg.png"/pic:cNvPicPra:picLocks noChangeAspect="1" noChangeArrowheads="1"//pic:cNvPicPr/pic:nvPicPrpic:blipFilla:blip r:embed="rId7"a:extLsta:ext uri="{28A0092B-C50C-407E-A947-70E740481C1C}"a14:useLocalDpi xmlns:a14="http://schemas.microsoft.com/office/drawing/2010/main" val="0"//a:ext/a:extLst/a:blipa:srcRect/a:stretcha:fillRect//a:stretch/pic:blipFillpic:spPr bwMode="auto"a:xfrma:off x="0" y="0"/a:ext cx="1548765" cy="1409700"//a:xfrma:prstGeom prst="rect"a:avLst//a:prstGeoma:noFill/a:lna:noFill//a:ln/pic:spPr/pic:pic/a:graphicData/a:graphicwp14:sizeRelH relativeFrom="page"wp14:pctWidth0/wp14:pctWidth/wp14:sizeRelHwp14:sizeRelV relativeFrom="page"wp14:pctHeight0/wp14:pctHeight/wp14:sizeRelV/wp:anchor/w:drawing/w:r/w:pw:p w:rsidR="00433D93" w:rsidRPr="00062DDC" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"/w:rPrw:b/w:sz w:val="24"//w:rPr/w:pPrw:r w:rsidRPr="00062DDC"w:rPrw:rFonts w:hint="eastAsia"/w:b/w:sz w:val="24"//w:rPrw:t1. Introduction/w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"//w:pPrw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:tThe P versus NP problem is one of the most important problems in computer science. In 1971, Stephan Cook proved that the 3-SAT is NP-complete [1], which means that if 3-SAT can solved in polynomial time, we can state that P=NP./w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"//w:pPr/w:pw:p w:rsidR="00433D93" w:rsidRPr="00062DDC" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"/w:rPrw:b/w:sz w:val="28"/w:szCs w:val="28"//w:rPr/w:pPrw:r w:rsidRPr="00F11793"w:rPrw:rFonts w:hint="eastAsia"/w:b/w:sz w:val="24"/w:szCs w:val="28"//w:rPrw:t2. Preliminaries/w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"//w:pPrw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve"See /w:t/w:rw:rw:tWikipedia/w:t/w:rw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve": /w:t/w:rw:rw:thttp://en.wikipedia.org/wiki/Boolean_satisfiability_problem/w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"//w:pPr/w:pw:p w:rsidR="00433D93" w:rsidRPr="00F11793" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"/w:rPrw:b/w:sz w:val="24"/w:szCs w:val="28"//w:rPr/w:pPrw:r w:rsidRPr="00F11793"w:rPrw:rFonts w:hint="eastAsia"/w:b/w:sz w:val="24"/w:szCs w:val="28"//w:rPrw:t xml:space="preserve"3. Main /w:t/w:rw:r w:rsidRPr="00F11793"w:rPrw:b/w:sz w:val="24"/w:szCs w:val="28"//w:rPrw:ttheorem/w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRPr="00D21195" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"/w:rPrw:b//w:rPr/w:pPrw:r w:rsidRPr="00D21195"w:rPrw:rFonts w:hint="eastAsia"/w:b//w:rPrw:tTheorem 1/w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRPr="009E3972" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"/w:rPrw:i//w:rPr/w:pPrw:r w:rsidRPr="009E3972"w:rPrw:rFonts w:hint="eastAsia"/w:i//w:rPrw:t xml:space="preserve"3-SAT can be solved /w:t/w:rw:proofErr w:type="gramStart"/w:r w:rsidRPr="009E3972"w:rPrw:i//w:rPrw:tin/w:t/w:rw:r w:rsidRPr="009E3972"w:rPrw:rFonts w:hint="eastAsia"/w:i//w:rPrw:t xml:space="preserve" /w:t/w:rw:proofErr w:type="gramEnd"/m:oMathm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tO(n)/m:t/m:r/m:oMathw:r w:rsidRPr="009E3972"w:rPrw:rFonts w:hint="eastAsia"/w:i//w:rPrw:t./w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"//w:pPrw:proofErr w:type="gramStart"/w:r w:rsidRPr="00D21195"w:rPrw:rFonts w:hint="eastAsia"/w:b//w:rPrw:tProof./w:t/w:rw:proofErr w:type="gramEnd"/w:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve" Let us denote given 3-SAT /w:t/w:rm:oMathm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tE/m:t/m:r/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve" as follows:/w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"/w:jc w:val="center"//w:pPrm:oMathm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tE=/m:t/m:rm:dm:dPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:dPrm:em:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:sSubPrm:em:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t11/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t12/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t13/m:t/m:r/m:sub/m:sSubm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:e/m:dm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∧/m:t/m:rm:dm:dPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:dPrm:em:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t21/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t22/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t23/m:t/m:r/m:sub/m:sSub/m:e/m:dm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∧…∧/m:t/m:rm:dm:dPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:dPrm:em:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tn1/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tn2/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tn3/m:t/m:r/m:sub/m:sSub/m:e/m:d/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t,/w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"//w:pPrw:proofErr w:type="gramStart"/w:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve"where /w:t/w:rw:proofErr w:type="gramEnd"/m:oMathm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:sSubPrm:em:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tij/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t=/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tx/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tk/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t xml:space="preserve" or ¬/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tx/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tk/m:t/m:r/m:sub/m:sSub/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve". Since a clause /w:t/w:rm:oMathm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t(/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:sSubPrm:em:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tk1/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tk2/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tk3/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t)/m:t/m:r/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve" is equals /w:t/w:rw:proofErr w:type="gramStart"/w:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve"to /w:t/w:rw:proofErr w:type="gramEnd"/m:oMathm:dm:dPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:dPrm:em:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:sSubPrm:em:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tk1/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tk2/m:t/m:r/m:sub/m:sSubm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:e/m:dm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨(/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tk2/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tk3/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t)/m:t/m:r/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve", we can reform /w:t/w:rm:oMathm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tE/m:t/m:r/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve" as follows:/w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"/w:jc w:val="center"//w:pPrm:oMathm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tE=/m:t/m:rm:dm:dPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:dPrm:em:dm:dPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:dPrm:em:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:sSubPrm:em:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t11/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t12/m:t/m:r/m:sub/m:sSubm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:e/m:dm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:dm:dPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:dPrm:em:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t12/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t13/m:t/m:r/m:sub/m:sSub/m:e/m:dm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:e/m:dm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∧…∧(/m:t/m:rm:dm:dPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:dPrm:em:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tn1/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tn2/m:t/m:r/m:sub/m:sSub/m:e/m:dm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:dm:dPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:dPrm:em:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tn2/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ta/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tn3/m:t/m:r/m:sub/m:sSub/m:e/m:dm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t)/m:t/m:r/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t./w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"//w:pPrw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:tBy replacing clauses with variables (in case some clauses are equal, replace them with the same variables), we can obtain 2-/w:t/w:rw:proofErr w:type="gramStart"/w:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve"SAT /w:t/w:rw:proofErr w:type="gramEnd"/m:oMathm:sSupm:sSupPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:sSupPrm:em:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tE/m:t/m:r/m:em:supm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t'/m:t/m:r/m:sup/m:sSup/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve". The number of clauses of /w:t/w:rm:oMathm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tE'/m:t/m:r/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve" /w:t/w:rw:proofErr w:type="gramStart"/w:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve"is /w:t/w:rw:proofErr w:type="gramEnd"/m:oMathm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tn/m:t/m:r/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve", and the number of variables in /w:t/w:rm:oMathm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tE'/m:t/m:r/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve" is at most /w:t/w:rm:oMathm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t2n/m:t/m:r/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve". It is known that 2-SAT can be solved in linear time [2]. /w:t/w:rw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t■/w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRPr="0081440D" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"//w:pPrw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:tFor instance, for 3-/w:t/w:rw:proofErr w:type="gramStart"/w:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve"SAT /w:t/w:rw:proofErr w:type="gramEnd"/m:oMathm:dm:dPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:dPrm:em:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:sSubPrm:em:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tx/m:t/m:r/m:em:subm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t1/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨¬/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tx/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t2/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tx/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t3/m:t/m:r/m:sub/m:sSubm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:e/m:dm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∧(/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tx/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t1/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨¬/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tx/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t2/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tx/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t4/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t)/m:t/m:r/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve", the corresponding 2-SAT is /w:t/w:rm:oMathm:dm:dPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:dPrm:em:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:sSubPrm:em:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ty/m:t/m:r/m:em:subm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t1/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ty/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t2/m:t/m:r/m:sub/m:sSubm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:e/m:dm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∧(/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ty/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t1/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ty/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t3/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t)/m:t/m:r/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve", where /w:t/w:rm:oMathm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:sSubPrm:em:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ty/m:t/m:r/m:em:subm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t1/m:t/m:r/m:sub/m:sSub/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve", /w:t/w:rm:oMathm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:sSubPrm:em:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ty/m:t/m:r/m:em:subm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t2/m:t/m:r/m:sub/m:sSub/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve" and /w:t/w:rm:oMathm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:sSubPrm:em:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:ty/m:t/m:r/m:em:subm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t3/m:t/m:r/m:sub/m:sSub/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve" are corresponding to /w:t/w:rm:oMathm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t(/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:sSubPrm:em:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tx/m:t/m:r/m:em:subm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t1/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨¬/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tx/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t2/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t)/m:t/m:r/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve", /w:t/w:rm:oMathm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t(¬/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:sSubPrm:em:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tx/m:t/m:r/m:em:subm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t2/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tx/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t3/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t)/m:t/m:r/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve" and /w:t/w:rm:oMathm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t(¬/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPr/m:ctrlPr/m:sSubPrm:em:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tx/m:t/m:r/m:em:subm:rm:rPrm:sty m:val="p"//m:rPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t2/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t∨/m:t/m:rm:sSubm:sSubPrm:ctrlPrw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/w:i//w:rPr/m:ctrlPr/m:sSubPrm:em:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:tx/m:t/m:r/m:em:subm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t4/m:t/m:r/m:sub/m:sSubm:rw:rPrw:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"//w:rPrm:t)/m:t/m:r/m:oMathw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t, respectively./w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"//w:pPr/w:pw:p w:rsidR="00433D93" w:rsidRPr="00F11793" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"/w:rPrw:b/w:sz w:val="24"/w:szCs w:val="28"//w:rPr/w:pPrw:r w:rsidRPr="00F11793"w:rPrw:rFonts w:hint="eastAsia"/w:b/w:sz w:val="24"/w:szCs w:val="28"//w:rPrw:lastRenderedPageBreak/w:t4. Conclusion/w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"//w:pPrw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve"Give me $1,000,000!!! /w:t/w:rw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:tщ/w:t/w:rw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t(/w:t/w:rw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:tﾟДﾟщ/w:t/w:rw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t)/w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"//w:pPr/w:pw:p w:rsidR="00433D93" w:rsidRPr="00F11793" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"/w:rPrw:b/w:sz w:val="24"/w:szCs w:val="28"//w:rPr/w:pPrw:r w:rsidRPr="00F11793"w:rPrw:rFonts w:hint="eastAsia"/w:b/w:sz w:val="24"/w:szCs w:val="28"//w:rPrw:tReferences/w:t/w:r/w:pw:p w:rsidR="00433D93" w:rsidRDefault="00433D93" w:rsidP="00433D93"w:pPrw:spacing w:line="360" w:lineRule="auto"//w:pPrw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t1. Cook, S. A.: The complexity of theorem-proving procedures. Proceedings of the 3/w:t/w:rw:r w:rsidRPr="001A06AB"w:rPrw:rFonts w:hint="eastAsia"//w:rPrw:trd/w:t/w:rw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve" Annual ACM Symposium on Theory of Computing (1971) 151-158/w:t/w:r/w:pw:p w:rsidR="00587A68" w:rsidRPr="002C7211" w:rsidRDefault="00433D93" w:rsidP="00EB523D"w:pPrw:spacing w:line="360" w:lineRule="auto"//w:pPrw:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve"2. /w:t/w:rw:proofErr w:type="spellStart"/w:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:tKrom/w:t/w:rw:proofErr w:type="spellEnd"/w:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve", L. R.: The Decision Problem for a Class of First-Order Formulas in /w:t/w:rw:proofErr w:type="gramStart"/w:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:tWhich/w:t/w:rw:proofErr w:type="gramEnd"/w:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve" all /w:t/w:rw:proofErr w:type="spellStart"/w:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:tDisjnctions/w:t/w:rw:proofErr w:type="spellEnd"/w:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve" are Binary. /w:t/w:rw:proofErr w:type="spellStart"/w:proofErr w:type="gramStart"/w:r w:rsidRPr="00CF00B8"w:teitschrift/w:t/w:rw:proofErr w:type="spellEnd"/w:proofErr w:type="gramEnd"/w:r w:rsidRPr="00CF00B8"w:t xml:space="preserve" /w:t/w:rw:proofErr w:type="spellStart"/w:r w:rsidRPr="00CF00B8"w:tfür/w:t/w:rw:proofErr w:type="spellEnd"/w:r w:rsidRPr="00CF00B8"w:t xml:space="preserve" /w:t/w:rw:proofErr w:type="spellStart"/w:r w:rsidRPr="00CF00B8"w:tMathematische/w:t/w:rw:proofErr w:type="spellEnd"/w:r w:rsidRPr="00CF00B8"w:t xml:space="preserve" /w:t/w:rw:proofErr w:type="spellStart"/w:r w:rsidRPr="00CF00B8"w:tLogik/w:t/w:rw:proofErr w:type="spellEnd"/w:r w:rsidRPr="00CF00B8"w:t xml:space="preserve" und /w:t/w:rw:proofErr w:type="spellStart"/w:r w:rsidRPr="00CF00B8"w:tGrundlagen/w:t/w:rw:proofErr w:type="spellEnd"/w:r w:rsidRPr="00CF00B8"w:t xml:space="preserve" der /w:t/w:rw:proofErr w:type="spellStart"/w:r w:rsidRPr="00CF00B8"w:tMathematik/w:t/w:rw:proofErr w:type="spellEnd"/w:rw:rPrw:rFonts w:hint="eastAsia"//w:rPrw:t xml:space="preserve" 13 (1967) 15-20/w:t/w:r/w:pw:sectPr w:rsidR="00587A68" w:rsidRPr="002C7211" w:rsidSect="005C46F8"w:headerReference w:type="even" r:id="rId8"/w:headerReference w:type="default" r:id="rId9"/w:footerReference w:type="even" r:id="rId10"/w:footerReference w:type="default" r:id="rId11"/w:headerReference w:type="first" r:id="rId12"/w:footerReference w:type="first" r:id="rId13"/w:pgSz w:w="11906" w:h="16838"/w:pgMar w:top="1985" w:right="1701" w:bottom="1701" w:left="1701" w:header="851" w:footer="992" w:gutter="0"/w:cols w:space="425"/w:docGrid w:linePitch="360"//w:sectPr/w:body/w:document
()
​
